# ImmunoTherapy — Baseline Model: Logistic Regression

Predicting **irAE occurrence** (`Grade 2+` = 1 vs `Grade 0-1` = 0) from the Khan
*JITC* 2025 cohort: 146 patients, 40 baseline cytokines + demographics + ANA titer.

A **basic baseline** (L2 logistic regression, standard preprocessing, repeated
stratified CV), followed by an **independent from-scratch check** and a
**log-scaled-cytokine** version.

## 1. Load data
The repo is public, so we read the CSV straight from GitHub — no upload needed.

In [ ]:
import pandas as pd
import numpy as np

RAW_URL = "https://raw.githubusercontent.com/vishnusrikant/ImmunoTherapy/main/ActionableData/Khan_workable.csv"
df = pd.read_csv(RAW_URL)

print("shape:", df.shape)
df.head(3)

## 2. Sanity check
Confirm the target is clean 0/1 and the cytokines loaded as numeric.

In [ ]:
TARGET = "iraE_occurrence"
print("target balance:")
print(df[TARGET].value_counts(), "\n")
print("dtypes summary:")
print(df.dtypes.value_counts())
print("\nmissing values per column (only showing >0):")
print(df.isna().sum()[df.isna().sum() > 0])

## 3. Define features & build the pipeline

- **Numeric** (40 cytokines + 2 ANA columns): median-impute (ANA is ~62% missing) → standardize.
- **Categorical** (gender, ethnicity, race, cancer_type, ici_drug): one-hot encode.
- **Model**: `LogisticRegression` (L2, the sklearn default) with `max_iter` raised so it converges.

Everything lives inside one `Pipeline` so preprocessing is **refit inside each CV fold** —
no leakage of scaler/imputer statistics from validation into training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

ID_COL   = "patient_id"
CAT_COLS = ["gender", "ethnicity", "race", "cancer_type", "ici_drug"]
ANA_COLS = ["ana_titer_baseline", "ana_titer_missing"]
# everything else (besides id/target/cat/ana) is a cytokine measurement
CYTO_COLS = [c for c in df.columns if c not in [ID_COL, TARGET] + CAT_COLS + ANA_COLS]
NUM_COLS  = CYTO_COLS + ANA_COLS

print(f"{len(CYTO_COLS)} cytokines + {len(ANA_COLS)} ANA = {len(NUM_COLS)} numeric, "
      f"{len(CAT_COLS)} categorical")

X = df[NUM_COLS + CAT_COLS]
y = df[TARGET].astype(int)

numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical = OneHotEncoder(handle_unknown="ignore")

pre = ColumnTransformer([
    ("num", numeric,     NUM_COLS),
    ("cat", categorical, CAT_COLS),
])

clf = Pipeline([
    ("pre", pre),
    ("lr",  LogisticRegression(max_iter=5000, random_state=42)),
])
clf

## 4. Cross-validated performance

With only 146 rows a single train/test split is too noisy to trust, so we use
**RepeatedStratifiedKFold (5 folds × 10 repeats = 50 fits)** and report the mean ± std
of ROC-AUC, PR-AUC (average precision), and accuracy.

Reference points: a no-skill classifier scores **ROC-AUC 0.50**, and always-predicting
the majority class gives **accuracy ≈ 0.54** (79/146).

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

for metric in ["roc_auc", "average_precision", "accuracy"]:
    scores = cross_val_score(clf, X, y, cv=cv, scoring=metric)
    print(f"{metric:18s}: {scores.mean():.3f} +/- {scores.std():.3f}")

## 5. Confusion matrix & classification report
Using out-of-fold predictions (each patient predicted by a model that never saw them).

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(clf, X, y, cv=skf)

print(classification_report(y, y_pred, target_names=["Grade 0-1", "Grade 2+"]))

ConfusionMatrixDisplay.from_predictions(
    y, y_pred, display_labels=["Grade 0-1", "Grade 2+"], cmap="Blues")
plt.title("Out-of-fold confusion matrix")
plt.show()

## 6. ROC curve (out-of-fold probabilities)

In [ ]:
from sklearn.metrics import RocCurveDisplay

y_proba = cross_val_predict(clf, X, y, cv=skf, method="predict_proba")[:, 1]
RocCurveDisplay.from_predictions(y, y_proba, name="Logistic Regression")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="No skill")
plt.title("ROC — out-of-fold")
plt.legend()
plt.show()

## 7. Which features carry the signal?
Refit on all 146 patients and read the standardized coefficients (log-odds). Positive =
pushes toward `Grade 2+`. With correlated cytokines, treat individual coefficients as
*suggestive*, not definitive.

In [ ]:
clf.fit(X, y)
feat_names = clf.named_steps["pre"].get_feature_names_out()
coefs = clf.named_steps["lr"].coef_[0]

coef_df = (pd.DataFrame({"feature": feat_names, "coef": coefs})
           .assign(abs_coef=lambda d: d["coef"].abs())
           .sort_values("abs_coef", ascending=False))

print("Top 15 features by |coefficient|:")
coef_df.head(15)[["feature", "coef"]]

## 8. Independent check — logistic regression from scratch (no sklearn)

A dependency-free reimplementation that reproduces Section 4's headline numbers using
only the Python standard library: it one-hot encodes, standardizes inside each fold,
fits logistic regression by gradient descent, and scores out-of-fold predictions with a
hand-computed ROC-AUC. If this lands near the sklearn result (~0.60 AUC), the baseline is
confirmed independently — not an artifact of any one library.

In [ ]:
import math, random

y_list = df[TARGET].astype(int).tolist()
num_mat = df[NUM_COLS].to_numpy().tolist()              # floats; NaN for missing ANA
cat_levels = {c: sorted(df[c].dropna().unique().tolist()) for c in CAT_COLS}
cat_cols   = [(c, lv) for c in CAT_COLS for lv in cat_levels[c]]
cat_mat    = df[CAT_COLS].to_dict("records")

def build_row(i):
    row = [None if (v != v) else float(v) for v in num_mat[i]]   # v!=v => NaN
    row += [1.0 if cat_mat[i][c] == lv else 0.0 for c, lv in cat_cols]
    return row

Xall  = [build_row(i) for i in range(len(df))]
n, p  = len(Xall), len(Xall[0])
num_p = len(NUM_COLS)

def auc(scores, labels):
    pairs = sorted(zip(scores, labels)); ranks = [0]*len(pairs); i = 0
    while i < len(pairs):                                   # average ranks for ties
        j = i
        while j+1 < len(pairs) and pairs[j+1][0] == pairs[i][0]: j += 1
        for k in range(i, j+1): ranks[k] = (i+j)/2.0 + 1
        i = j+1
    pos = sum(labels); neg = len(labels)-pos
    sr  = sum(ranks[k] for k in range(len(pairs)) if pairs[k][1] == 1)
    return (sr - pos*(pos+1)/2.0) / (pos*neg)

def train_lr(Xtr, ytr, l2=1.0, lr=0.1, iters=600):
    w = [0.0]*p; b = 0.0; m = len(Xtr)
    for _ in range(iters):
        gw = [0.0]*p; gb = 0.0
        for xi, yi in zip(Xtr, ytr):
            z  = b + sum(w[k]*xi[k] for k in range(p))
            pr = 1/(1+math.exp(-max(-30, min(30, z))))
            d  = pr - yi
            for k in range(p): gw[k] += d*xi[k]
            gb += d
        for k in range(p): w[k] -= lr*((gw[k]/m) + (l2/m)*w[k])
        b -= lr*gb/m
    return w, b

def predict(w, b, xi):
    z = b + sum(w[k]*xi[k] for k in range(p))
    return 1/(1+math.exp(-max(-30, min(30, z))))

def strat_folds(yv, k, seed):
    random.seed(seed)
    pos = [i for i in range(len(yv)) if yv[i] == 1]
    neg = [i for i in range(len(yv)) if yv[i] == 0]
    random.shuffle(pos); random.shuffle(neg)
    folds = [[] for _ in range(k)]
    for i, ix in enumerate(pos): folds[i % k].append(ix)
    for i, ix in enumerate(neg): folds[i % k].append(ix)
    return folds

K, REPEATS = 5, 4
aucs, accs = [], []
for rep in range(REPEATS):
    folds = strat_folds(y_list, K, seed=rep)
    oof_s = [0.0]*n; oof_p = [0]*n
    for f in range(K):
        test  = set(folds[f]); train = [i for i in range(n) if i not in test]
        means = [0.0]*num_p; sds = [1.0]*num_p; imp = [0.0]*num_p
        for k in range(num_p):
            vals = [Xall[i][k] for i in train if Xall[i][k] is not None]
            mu   = sum(vals)/len(vals); imp[k] = means[k] = mu
            var  = sum((v-mu)**2 for v in vals)/max(1, len(vals)-1)
            sds[k] = math.sqrt(var) if var > 0 else 1.0
        def std_row(i):
            r = list(Xall[i])
            for k in range(num_p):
                v = r[k] if r[k] is not None else imp[k]
                r[k] = (v-means[k])/sds[k]
            return r
        Xtr = [std_row(i) for i in train]; ytr = [y_list[i] for i in train]
        w, b = train_lr(Xtr, ytr)
        for i in test:
            s = predict(w, b, std_row(i)); oof_s[i] = s; oof_p[i] = 1 if s >= 0.5 else 0
    aucs.append(auc(oof_s, y_list))
    accs.append(sum(1 for i in range(n) if oof_p[i] == y_list[i])/n)

print(f"n={n} patients, p={p} features  (majority-class acc = {max(sum(y_list), n-sum(y_list))/n:.3f})")
print(f"ROC-AUC : {sum(aucs)/len(aucs):.3f}   per-repeat: {[round(a,3) for a in aucs]}")
print(f"Accuracy: {sum(accs)/len(accs):.3f}   per-repeat: {[round(a,3) for a in accs]}")

## 9. Improvement — log-scale the cytokines

The 40 cytokines span ~6 orders of magnitude (≈0.03 → 171,000 pg/mL). On that raw scale a
linear model is dominated by a few large-valued markers. We apply **`log1p`** to the
cytokines *inside the pipeline* (so it's refit per fold, and the raw CSV is untouched),
then standardize. ANA and categoricals are handled exactly as before. Compare the ROC-AUC
below against Section 4 to see whether the log scale helps.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

# cytokines: log1p -> standardize ;  ANA: median-impute -> standardize ;  cats: one-hot
cyto_pipe = Pipeline([
    ("log",   FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])
ana_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])

pre_log = ColumnTransformer([
    ("cyto", cyto_pipe, CYTO_COLS),
    ("ana",  ana_pipe,  ANA_COLS),
    ("cat",  OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
])

clf_log = Pipeline([
    ("pre", pre_log),
    ("lr",  LogisticRegression(max_iter=5000, random_state=42)),
])

print("Log-scaled cytokines — RepeatedStratifiedKFold (5x10):")
for metric in ["roc_auc", "average_precision", "accuracy"]:
    scores = cross_val_score(clf_log, X, y, cv=cv, scoring=metric)
    print(f"{metric:18s}: {scores.mean():.3f} +/- {scores.std():.3f}")

In [ ]:
# Side-by-side: raw vs log-scaled cytokines (ROC-AUC)
raw_auc = cross_val_score(clf,     X, y, cv=cv, scoring="roc_auc").mean()
log_auc = cross_val_score(clf_log, X, y, cv=cv, scoring="roc_auc").mean()
print(f"ROC-AUC  raw cytokines : {raw_auc:.3f}")
print(f"ROC-AUC  log cytokines : {log_auc:.3f}")
print(f"delta                  : {log_auc - raw_auc:+.3f}")

## Next steps
1. ~~Log-transform the cytokines~~ — **done in Section 9.**
2. **Elastic-net penalty** (`penalty="elasticnet", solver="saga", l1_ratio=...`) with a
   CV grid — adds feature selection and stabilizes correlated cytokines.
3. **Compare against HistGradientBoosting** to see the non-linear ceiling.
4. Collapse rare `ici_drug` / `cancer_type` levels to cut one-hot noise.